# Israeli Supreme Court case law → chunks, embeddings, offline indexes

Runs `caselaw/israeli_caselaw_ingest` end to end on Colab (free tier: T4, ~12 GB RAM):

1. mount Google Drive: every output lives under `ROOT` on Drive;
2. install the pinned dependencies and the package;
3. download `cases_all.parquet` from the Hugging Face dataset `LevMuchnik/SupremeCourtOfIsrael` (the only data source; nothing is scraped);
4. check the file's real column list;
5. **dry run** on 2,000 documents (filter report + 5 random chunks);
6. the **full run**, stage by stage, then `validate`;
7. zip the artifacts for the offline machine.

**Every cell resumes.** Each stage writes done-markers and skips finished work, and embedding checkpoints every 20,000 chunks. After a disconnect, run the setup cells (1–3) again, then re-run the cell that stopped.

Runtime → Change runtime type → **T4 GPU** (only the embed stage needs it).

## 1. Settings and Google Drive

In [ ]:
# --- settings -------------------------------------------------------------------------------
ROOT = "/content/drive/MyDrive/israeli_caselaw"      # outputs + dataset cache (on Drive)
REPO = "zananiri/AI-IZ"
BRANCH = "main"              # the branch to run
SAMPLE_N = 2000                                      # dry-run size
EMBED_MODEL = "BAAI/bge-m3"                          # or "intfloat/multilingual-e5-large"
EMBED_YES = False                                    # True = embed even if the estimate is over 3 h
# ---------------------------------------------------------------------------------------------
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(ROOT, exist_ok=True)
os.environ["CASELAW_ROOT"] = ROOT
print("outputs ->", ROOT)

## 2. Get the code and install

If the repository is private, add a GitHub token with read access as a Colab secret named `GITHUB_TOKEN` (the key icon in the left bar). The code is cloned to `/content` (not Drive), so this cell is cheap to re-run.

In [ ]:
import os, subprocess
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if not os.path.exists("/content/AI-IZ/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, "/content/AI-IZ"], check=True)
else:
    subprocess.run(["git", "-C", "/content/AI-IZ", "pull", "--ff-only"], check=True)
%cd /content/AI-IZ/caselaw
!pip install -q -r requirements.txt
!pip install -q --no-deps -e .

In [ ]:
# Settings for this session: config.yaml plus the choices above, written next to the outputs.
import yaml, pathlib
overrides = {"embed": {"model": EMBED_MODEL}}
CONFIG = pathlib.Path(ROOT) / "colab_config.yaml"
CONFIG.write_text(yaml.safe_dump(overrides, allow_unicode=True))
os.environ["CFG"] = str(CONFIG)
os.environ["SAMPLE_N"] = str(SAMPLE_N)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "no GPU: fine until the embed stage"

In [ ]:
# Unit tests (encoding repair, dates incl. the bogus-1920 case, header parsing, chunk boundaries and
# overlap, citation regex, stable chunk ids) and an end-to-end run on synthetic data. ~1 min.
!python -m pytest -q

## 3. Download the dataset (~1.5 GB, once)

In [ ]:
!python -m israeli_caselaw_ingest download --config "$CFG"

## 4. Check the real columns

The pipeline maps each field to the columns the file actually has (first non-empty wins) and stops if `text`, `Type` or every date column is missing. This cell shows the mapping, the distinct `Type` values and a few rows, so you can compare them with the expected layout before relying on it.

In [ ]:
import pyarrow.parquet as pq, pyarrow.compute as pc
from israeli_caselaw_ingest.config import load_config
from israeli_caselaw_ingest.download import dataset_path
from israeli_caselaw_ingest import schema

cfg = load_config(os.environ["CFG"])
src = dataset_path(cfg)
cols = schema.check_schema(src)
print(schema.schema_report(src, cols))

pf = pq.ParquetFile(src)
type_col = cols["type"][0]
types = pc.value_counts(pq.read_table(src, columns=[type_col]).column(0)).to_pylist()
print(f"\n{type_col} values:")
for t in sorted(types, key=lambda t: -t["counts"]):
    print(f"  {t['counts']:>8,}  {t['values']}")

sample_cols = [c for c in pf.schema_arrow.names if c != "text"]
first = next(pf.iter_batches(batch_size=3, columns=sample_cols + ["text"])).to_pylist()
for row in first:
    print("\n" + "-" * 80)
    for k in sample_cols:
        print(f"{k:>20}: {str(row.get(k))[:120]}")
    print(f"{'text':>20}: {str(row.get('text'))[:300]!r}")

## 5. Dry run: 2,000 documents

Everything (filter → clean → chunk → BM25 → embed → validate) on the first 2,000 documents that pass the filter, written to `ROOT/sample_2000/` so it never mixes with the full run. The embed step times 2,000 chunks and prints an estimate for the sample.

In [ ]:
!python -m israeli_caselaw_ingest all --config "$CFG" --sample-n $SAMPLE_N --skip-download --embed --yes

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{ROOT}/sample_{SAMPLE_N}/reports/filter_report.md", encoding="utf-8").read()))

In [ ]:
# Five random chunks: check that the Hebrew reads correctly and the headers/holdings look right.
!python -m israeli_caselaw_ingest peek --config "$CFG" --sample-n $SAMPLE_N --n 5

## 6. Full run

One cell per stage; re-run a cell after a disconnect and it continues. Rough costs on free Colab: filter ~10–20 min (reads 1.5 GB), clean and chunk up to an hour or so each (CPU), BM25 tens of minutes, and **embed several hours**. The embed cell prints its estimate first and stops above 3 h unless `EMBED_YES = True`. Because it checkpoints per shard, a long embed can be spread over several sessions.

In [ ]:
!python -m israeli_caselaw_ingest filter --config "$CFG"
display(Markdown(open(f"{ROOT}/reports/filter_report.md", encoding="utf-8").read()))

In [ ]:
!python -m israeli_caselaw_ingest clean --config "$CFG"

In [ ]:
!python -m israeli_caselaw_ingest chunk --config "$CFG"

In [ ]:
!python -m israeli_caselaw_ingest bm25 --config "$CFG"

In [ ]:
yes = "--yes" if EMBED_YES else ""
!python -m israeli_caselaw_ingest embed --config "$CFG" {yes}

In [ ]:
!python -m israeli_caselaw_ingest validate --config "$CFG" > /dev/null
display(Markdown(open(f"{ROOT}/reports/validation_report.md", encoding="utf-8").read()))

## 7. Zip the artifacts for the offline machine

Writes `ROOT/israeli_caselaw_artifacts.zip` with `chunks/`, `documents.parquet`, `lancedb/` (or `faiss/`), `bm25/`, `reports/` and the saved embedding model (`models/`, needed to encode queries offline). Parquet and vectors are already compressed, so the zip only stores them (`-0`), which is fast. For the full corpus expect tens of GB; you can also copy the folders straight from Drive.

In [ ]:
%cd {ROOT}
parts = [p for p in ["chunks", "documents.parquet", "lancedb", "faiss", "bm25", "reports", "models", "embeddings/id_map.parquet"]
         if os.path.exists(p)]
print("zipping:", parts)
!rm -f israeli_caselaw_artifacts.zip && zip -q -r -0 israeli_caselaw_artifacts.zip {" ".join(parts)} && ls -lh israeli_caselaw_artifacts.zip
%cd /content/AI-IZ/caselaw

## 8. Offline use (on the other machine)

```python
from israeli_caselaw_ingest.config import load_config
from israeli_caselaw_ingest.search import Searcher
s = Searcher(load_config(root="/path/to/unzipped"))
hits = s.hybrid("עקרון המידתיות", k=5)             # BM25 + dense, reciprocal-rank fusion
rows = s.rows([cid for cid, _ in hits])            # citation, date, text, source_url
```

Keep the attribution: every chunk has `source_dataset` and `source_url`, and please cite the dataset's curators (Muchnik, Yahav, Nevo, Chriqui, Shektov, 2023). See `caselaw/README.md` for known gaps.